# Vector geometry toolkit

Build and reshape `FeatureCollection`s:

- **`from_bbox`** / **`from_records`** — construct collections from a box or from dict rows.
- **`with_coordinates`** / **`with_centroid`** — attach x/y or centroid columns.
- **`explode`** — split multi-part geometries into single parts.
- **`concat`** — stack two collections.

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-t3-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.feature import FeatureCollection
from shapely.geometry import Point

2026-06-08 23:58:38 | INFO | pyramids.base.config | Logging is configured.


## Construct — `from_bbox` and `from_records`

In [3]:
box = FeatureCollection.from_bbox((0.0, 0.0, 1.0, 1.0), epsg=4326)
pts = FeatureCollection.from_records(
    [{'id': 1, 'geometry': Point(0, 0)}, {'id': 2, 'geometry': Point(1, 1)}], crs=4326
)
(len(box), box.epsg), (len(pts), list(pts.columns))

((1, 4326), (2, ['id', 'geometry']))

## Derive columns — `with_coordinates` / `with_centroid`

In [4]:
polys = FeatureCollection.read_file(str(DATA / 'coello_polygons.geojson'))
coords = polys.with_coordinates()
centroids = polys.with_centroid()
[c for c in coords.columns if c in ('x', 'y')], type(centroids).__name__

(['x', 'y'], 'FeatureCollection')

## Reshape — `explode` and `concat`

In [5]:
single_parts = polys.explode()
stacked = polys.concat(polys)
(len(polys), '-> explode ->', len(single_parts)), ('concat ->', len(stacked))

((4, '-> explode ->', 4), ('concat ->', 8))

## Notes

- A `FeatureCollection` is a `geopandas.GeoDataFrame` subclass, so all of geopandas works too.
- See also: [Rasterize ↔ vectorize](rasterize-vectorize.ipynb),
  [Vector formats](../conversions/vector-formats.ipynb).